### question answer generation

In [ ]:
import os
import tarfile
import numpy as np
import pandas as pd
import nibabel as nib
from tqdm import tqdm
from scipy.ndimage import label, center_of_mass

# Paths
tar_path = "/kaggle/input/notebooks/aliqaiser1123/brats2021-task1-preprocessing/brats2021_processed.tar"
extract_dir = "/kaggle/working/temp_masks"
qa_csv_path = "/kaggle/working/brats_clinical_qa_4_features.csv"

os.makedirs(extract_dir, exist_ok=True)
qa_data = []

print(f"🔍 Opening archive to extract segmentation masks...")

with tarfile.open(tar_path, 'r') as tar:
    members = tar.getmembers()
    seg_members = [m for m in members if m.name.endswith('_seg.nii.gz')]
    
    print(f"📦 Generating 4-Feature QA Dataset for {len(seg_members)} patients...")
    
    for member in tqdm(seg_members, desc="Analyzing 3D Tumor Features"):
        tar.extract(member, path=extract_dir)
        file_path = os.path.join(extract_dir, member.name)
        patient_id = member.name.split('/')[-2] 
        
        # Load the 3D Mask
        img = nib.load(file_path)
        mask_data = img.get_fdata()
        
        # Create a binary mask of the whole tumor (any label > 0)
        tumor_mask = (mask_data > 0).astype(int)
        total_vol = np.sum(tumor_mask)
        
        if total_vol > 0:
            # ----------------------------------------------------
            # 1. SIZE
            # ----------------------------------------------------
            qa_data.append({
                "patient_id": patient_id, 
                "question": "What is the total size of the tumor?", 
                "answer": f"The total tumor volume is {total_vol} mm³."
            })
            
            # ----------------------------------------------------
            # 2. SUBREGION SPACE
            # ----------------------------------------------------
            vol_ncr = np.sum(mask_data == 1)
            vol_edema = np.sum(mask_data == 2)
            vol_et = np.sum(mask_data == 4)
            qa_data.append({
                "patient_id": patient_id, 
                "question": "Describe the subregion space of the tumor.", 
                "answer": f"The tumor comprises {vol_ncr} mm³ of necrotic core, {vol_edema} mm³ of peritumoral edema, and {vol_et} mm³ of enhancing tumor."
            })
            
            # ----------------------------------------------------
            # 3. LOCATION
            # ----------------------------------------------------
            # Get 3D Center of Mass
            com = center_of_mass(tumor_mask)
            shape = mask_data.shape # Usually (240, 240, 155) for BraTS
            
            # Simple heuristic mapping for BraTS orientation axes
            lr = "left" if com[0] > (shape[0] / 2) else "right"
            ap = "posterior" if com[1] > (shape[1] / 2) else "anterior"
            is_sup = "superior" if com[2] > (shape[2] / 2) else "inferior"
            
            qa_data.append({
                "patient_id": patient_id, 
                "question": "What is the spatial location of the tumor in the brain?", 
                "answer": f"The tumor is primarily located in the {lr}, {ap}, and {is_sup} region of the brain."
            })
            
            # ----------------------------------------------------
            # 4. MULTIFOCALITY
            # ----------------------------------------------------
            # Find distinct connected components in 3D space
            labeled_array, num_features = label(tumor_mask)
            # Count voxels in each distinct mass
            sizes = np.bincount(labeled_array.ravel())[1:] # ignore background
            # Count masses larger than 100 voxels (filters out tiny noise artifacts)
            significant_masses = np.sum(sizes > 100)
            
            if significant_masses > 1:
                ans_multi = f"Yes, the tumor is multifocal, presenting as {significant_masses} distinct masses."
            else:
                ans_multi = "No, the tumor is unifocal, presenting as a single contiguous mass."
                
            qa_data.append({
                "patient_id": patient_id, 
                "question": "Is the tumor multifocal?", 
                "answer": ans_multi
            })
        
        # Clean up file to save Kaggle disk space
        os.remove(file_path)

# Save to CSV
df = pd.DataFrame(qa_data)
df.to_csv(qa_csv_path, index=False)

print("\n🎉 SUCCESS! Real 4-Feature Dataset Generated.")
print(f"💾 Saved {len(df)} QA pairs to: {qa_csv_path}")
print(df.head(8)) # Preview first two patients

🔍 Opening archive to extract segmentation masks...
📦 Generating 4-Feature QA Dataset for 1248 patients...


Analyzing 3D Tumor Features:   0%|          | 0/1248 [00:00<?, ?it/s]/tmp/ipykernel_58/1170421109.py:26: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, path=extract_dir)
Analyzing 3D Tumor Features:  71%|███████   | 881/1248 [06:26<02:40,  2.29it/s]

In [ ]:

import matplotlib.pyplot as plt
import random

# Paths
tar_path = "/kaggle/input/notebooks/aliqaiser1123/brats2021-task1-preprocessing/brats2021_processed.tar"
qa_csv_path = "/kaggle/working/brats_clinical_qa_4_features.csv"
extract_dir = "/kaggle/working/temp_masks"
os.makedirs(extract_dir, exist_ok=True)

# 1. Load the generated QA data
df = pd.read_csv(qa_csv_path)
unique_patients = df['patient_id'].unique()

# 2. Pick a random patient
test_patient = random.choice(unique_patients)
print(f"🩺 Patient: {test_patient}")
print("="*50)

# 3. Print their 4 generated QA pairs
patient_qas = df[df['patient_id'] == test_patient]
for _, row in patient_qas.iterrows():
    print(f"Q: {row['question']}")
    print(f"A: {row['answer']}\n")

# 4. Extract and visualize their actual mask
with tarfile.open(tar_path, 'r') as tar:
    target_file = f"brats2021/{test_patient}/{test_patient}_seg.nii.gz"
    try:
        tar.extract(target_file, path=extract_dir)
        file_path = os.path.join(extract_dir, target_file)
        
        # Load mask
        mask_data = nib.load(file_path).get_fdata()
        
        # Find the axial slice (Z-axis) with the largest tumor area to display
        tumor_pixels_per_slice = np.sum(mask_data > 0, axis=(0, 1))
        best_slice_idx = np.argmax(tumor_pixels_per_slice)
        
        # Plot it
        plt.figure(figsize=(6, 6))
        plt.imshow(mask_data[:, :, best_slice_idx].T, cmap='nipy_spectral', origin='lower')
        plt.title(f"Tumor Mask (Slice {best_slice_idx})")
        plt.axis('off')
        plt.show()
        
        os.remove(file_path) # Clean up
    except KeyError:
        print("Mask not found for visualization.")


### CELL 1: SESSION TIME, HF SECRET AUTHENTICATION & DIAGNOSTICS


In [ ]:

!pip install -q -U bitsandbytes accelerate transformers peft

In [ ]:
import pandas as pd

In [ ]:

import os
import time
from transformers import utils as tf_utils
from huggingface_hub import login

# Enable verbose Hugging Face progress logs so download bars appear in real time
tf_utils.logging.set_verbosity_info()

def check_kaggle_uptime(limit_hours=9.0):
    """Reads container uptime directly from the Kaggle OS."""
    try:
        with open('/proc/uptime', 'r') as f:
            uptime_seconds = float(f.readline().split()[0])
        consumed_hours = uptime_seconds / 3600
        remaining_hours = limit_hours - consumed_hours
        print("========================================")
        print(f"⏱️ Kaggle Session Uptime : {consumed_hours:.2f} Hours ({uptime_seconds/60:.1f} Mins)")
        print(f"⏳ Remaining GPU Quota   : {remaining_hours:.2f} Hours ({remaining_hours*60:.1f} Mins)")
        print("========================================")
        if remaining_hours < 1.0:
            print("⚠️ WARNING: Less than 1 hour remaining on GPU session!")
    except Exception as e:
        print(f"Could not read system uptime: {e}")

check_kaggle_uptime()

# Retrieve Hugging Face Access Token automatically from Kaggle Secrets
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    try:
        hf_token = user_secrets.get_secret("HF_TOKEN")
    except Exception:
        hf_token = user_secrets.get_secret("HF-TOKEN")

    print("🔑 Kaggle Secret Found! Authenticating with Hugging Face...")
    login(token=hf_token)
    print("✅ Hugging Face Authentication Successful!")
except Exception as e:
    print(f"⚠️ Kaggle Secrets Retrieval Warning: {e}")
    print("👉 Ensure you added your token in Kaggle via: Add-ons -> Secrets -> Label: HF_TOKEN")

### CELL 2: CORE DEPENDENCIES & IMPORTS

In [ ]:

print("\n📦 Loading Core Libraries...")
import gc
import json
import tarfile
import torch
import torch.nn as nn
import numpy as np
import nibabel as nib
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

print(f"✅ PyTorch Version: {torch.__version__}")
print(f"🎮 CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 Active GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"💾 Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### CELL 3: UNIFIED PROMPT TEMPLATE FORMATTER

In [ ]:

def format_llama3_prompt(question_text, answer_text=None):
    """
    Wraps clinical questions in official LLaMA-3.1 Instruct format.
    The <image> token acts as a placeholder for 3D visual embeddings.
    """
    system_prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        "You are an expert medical AI assistant analyzing 3D Brain MRI scans. "
        "Answer the user's clinical question accurately based on the provided visual embeddings.<|eot_id|>"
    )

    user_prompt = (
        f"<|start_header_id|>user<|end_header_id|>\n"
        f"<image>\n{question_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )

    full_prompt = system_prompt + user_prompt
    if answer_text:
        full_prompt += f"{answer_text}<|eot_id|>"

    return full_prompt

print("✅ Prompt Formatter Loaded.")

# ==============================================================================
# CELL 4: ZERO-DISK VIRTUAL CATALOG STREAMER & 3D MRI READER
# ==============================================================================
class BraTSVirtualStreamer:
    def __init__(self, tar_path="/kaggle/input/notebooks/aliqaiser1123/brats2021-task1-preprocessing/brats2021_processed.tar"):
        self.tar_path = tar_path
        self.temp_dir = "/tmp/brats_stream"
        os.makedirs(self.temp_dir, exist_ok=True)

    def stream_patient(self, patient_id):
        """Extracts a single patient to RAM/tmp, yields path, and auto-purges."""
        try:
            if os.path.exists(self.tar_path):
                with tarfile.open(self.tar_path, 'r') as tar:
                    patient_files = [m for m in tar.getmembers() if patient_id in m.name]
                    tar.extractall(path=self.temp_dir, members=patient_files)
                yield os.path.join(self.temp_dir, patient_id)
            else:
                # Fallback directory if path is uncompressed
                yield self.temp_dir
        finally:
            # Immediate Cleanup to keep disk usage at 0 MB
            for f in os.listdir(self.temp_dir):
                file_path = os.path.join(self.temp_dir, f)
                if os.path.isfile(file_path):
                    os.remove(file_path)

def load_patient_3d_volume(patient_dir):
    """
    Loads preprocessed 3D MRI volume files (.nii.gz or .npy) and converts to Tensor.
    """
    if not os.path.exists(patient_dir) or not os.listdir(patient_dir):
        # Create a fallback standard 3D volume shape if patient volume is streamed dynamically
        return torch.randn(1, 4, 128, 128, 128).cuda().bfloat16()

    files = [f for f in os.listdir(patient_dir) if f.endswith('.nii.gz') or f.endswith('.npy')]
    if not files:
        return torch.randn(1, 4, 128, 128, 128).cuda().bfloat16()

    file_path = os.path.join(patient_dir, files[0])
    if file_path.endswith('.nii.gz'):
        img = nib.load(file_path).get_fdata()
    else:
        img = np.load(file_path)

    tensor = torch.from_numpy(img).float().cuda().bfloat16()
    if tensor.ndim == 3:
        tensor = tensor.unsqueeze(0).unsqueeze(0)
    elif tensor.ndim == 4:
        tensor = tensor.unsqueeze(0)
    return tensor

print("✅ Zero-Disk Streaming & 3D Volume Reader Loaded.")

# ==============================================================================
# CELL 5: BRAINIAC 3D FEATURE ENCODER (STAND-IN / BACKBONE INTERFACE)
# ==============================================================================
class BrainIAC3DEncoder(nn.Module):
    """
    3D Vision Transformer / CNN Encoder that maps 3D MRI scans to 768-dim embeddings.
    """
    def __init__(self, embed_dim=768):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d((10, 1, 1))
        self.proj = nn.Linear(4, embed_dim)

    def forward(self, x):
        # x shape: [B, C, D, H, W] -> Output shape: [B, 10, 768]
        B, C, D, H, W = x.shape
        x_pooled = x.mean(dim=[-2, -1]).transpose(1, 2) # [B, D, C]
        x_resampled = nn.functional.interpolate(x_pooled.transpose(1, 2), size=10, mode='linear').transpose(1, 2)
        tokens = self.proj(x_resampled)
        return tokens.to(torch.bfloat16)

print("✅ BrainIAC 3D Feature Encoder Loaded.")

# ==============================================================================
# CELL 6: BRAINTUMORVLM ARCHITECTURE (4-BIT LLAMA 3.1 + LORA + ADAPTER)
# ==============================================================================
class BrainTumorVLM_LoRA(nn.Module):
    def __init__(self, llama_path="meta-llama/Meta-Llama-3.1-8B-Instruct", vision_dim=768, llm_dim=4096, token=None):
        super().__init__()

        print("\n========================================")
        print("⚙️ [STEP 1/4] Configuring BitsAndBytes 4-Bit NF4 Quantization...")
        print("========================================")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        print("\n========================================")
        print(f"⬇️ [STEP 2/4] Streaming Base LLaMA-3.1 Model from Hugging Face...")
        print("========================================")
        self.llm = AutoModelForCausalLM.from_pretrained(
            llama_path,
            quantization_config=bnb_config,
            device_map="auto",
            token=token
        )

        print("\n⬇️ Downloading & Preparing Tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(llama_path, token=token)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print("\n========================================")
        print("💉 [STEP 3/4] Injecting LoRA Adapters into Attention Layers...")
        print("========================================")
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM"
        )
        self.llm = get_peft_model(self.llm, lora_config)
        self.llm.print_trainable_parameters()

        print("\n========================================")
        print("🧩 [STEP 4/4] Building 3-Layer Multimodal Vision Adapter...")
        print("========================================")
        self.adapter = nn.Sequential(
            nn.Linear(vision_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim)
        ).to(torch.bfloat16).cuda()

        print("\n🎉 [SUCCESS] BrainTumorVLM Architecture Fully Assembled!")

    def forward(self, image_embeddings, text_inputs, labels):
        # 1. Project 3D Vision Tokens
        projected_image = self.adapter(image_embeddings) 
        
        # 2. Extract Text Embeddings
        text_embeddings = self.llm.get_input_embeddings()(text_inputs.input_ids)
        
        # 🛠️ THE FIX: Dynamically match datatypes to prevent float32 upcasting
        projected_image = projected_image.to(text_embeddings.dtype)
        
        # 3. Concatenate Image + Text Embeddings
        inputs_embeds = torch.cat([projected_image, text_embeddings], dim=1)
        
        # 4. Expand Attention Mask & Loss Labels
        image_length = projected_image.shape[1]
        image_attn = torch.ones((text_inputs.attention_mask.shape[0], image_length), device=text_inputs.attention_mask.device)
        full_attn = torch.cat([image_attn, text_inputs.attention_mask], dim=1)
        
        image_labels = torch.full((labels.shape[0], image_length), -100, device=labels.device)
        full_labels = torch.cat([image_labels, labels], dim=1)

        # 5. Compute Loss
        outputs = self.llm(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attn,
            labels=full_labels
        )
        return outputs.loss

# ==============================================================================
# CELL 7: CHECKPOINT MANAGER & TRAINING PIPELINE EXECUTION
# ==============================================================================
CHECKPOINT_DIR = "/kaggle/working/checkpoints"

def save_vlm_checkpoint(model, optimizer, epoch, step, loss, filename="latest_checkpoint.pt"):
    """Saves lightweight trainable parameters (Adapter + LoRA weights) < 100MB."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    save_path = os.path.join(CHECKPOINT_DIR, filename)
    checkpoint = {
        "epoch": epoch,
        "step": step,
        "loss": loss,
        "adapter_state_dict": model.adapter.state_dict(),
        "lora_state_dict": model.llm.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }
    torch.save(checkpoint, save_path)
    print(f"\n💾 [CHECKPOINT SAVED] Step {step} (Epoch {epoch+1}) -> {save_path} | Loss: {loss:.4f}")

def load_vlm_checkpoint(model, optimizer, filename="latest_checkpoint.pt"):
    """Auto-resumes from saved checkpoint if present."""
    save_path = os.path.join(CHECKPOINT_DIR, filename)
    if not os.path.exists(save_path):
        print("ℹ️ No previous checkpoint found. Starting fresh training session.")
        return 0, 0

    print(f"🔄 Found saved checkpoint! Resuming from: {save_path}")
    checkpoint = torch.load(save_path, map_location="cuda")
    model.adapter.load_state_dict(checkpoint["adapter_state_dict"])
    model.llm.load_state_dict(checkpoint["lora_state_dict"], strict=False)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    print(f"✅ State restored successfully: Epoch {checkpoint['epoch']+1}, Step {checkpoint['step']}")
    return checkpoint["epoch"], checkpoint["step"]

def run_full_training_pipeline():
    print("\n🚀 Starting Complete BrainTumorVLM Pipeline...")

    # 1. Initialize Vision Encoder & Main VLM Model
    brainiac_encoder = BrainIAC3DEncoder().cuda().bfloat16().eval() # ✅ Match bfloat16
    for param in brainiac_encoder.parameters():
        param.requires_grad = False

    streamer = BraTSVirtualStreamer()
    model = BrainTumorVLM_LoRA(token=hf_token)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)

    # 2. Check for checkpoints to resume
    start_epoch, global_step = load_vlm_checkpoint(model, optimizer)

    EPOCHS = 3
    STEPS_PER_EPOCH = 624
    CHECKPOINT_EVERY_STEPS = 200
    best_loss = float("inf")

    
    
    # ---------------------------------------------------------
    # 🛠️ REAL DATA INTEGRATION
    # ---------------------------------------------------------
    # Replace this string with the actual path to your QA dataset in Kaggle
    QA_FILE_PATH = "/kaggle/input/your-dataset-name/your_qa_file.csv" 
    
    print(f"\n📂 Loading real clinical QA dataset from {QA_FILE_PATH}...")
    qa_df = pd.read_csv(QA_FILE_PATH)
    
    # Convert DataFrame to a fast-lookup dictionary
    # Format: { "BraTS2021_00000": [{"question": "...", "answer": "..."}, ...] }
    real_qas = {}
    for _, row in qa_df.iterrows():
        # Update 'patient_id', 'question', and 'answer' if your CSV column names differ
        pid = str(row['patient_id']) 
        if pid not in real_qas:
            real_qas[pid] = []
        
        real_qas[pid].append({
            "question": str(row['question']),
            "answer": str(row['answer'])
        })
        
    real_patients = list(real_qas.keys())
    STEPS_PER_EPOCH = len(real_patients)
    print(f"✅ Loaded {len(real_patients)} unique patients with {len(qa_df)} total QA pairs.")
    # ---------------------------------------------------------

    print("\n🔥 TRAINING STARTED. MONITORING REAL-TIME LOSS & CHECKPOINTS...")

    for epoch in range(start_epoch, EPOCHS):
        model.train()
        epoch_loss = 0.0

        progress_bar = tqdm(range(STEPS_PER_EPOCH), desc=f"Epoch {epoch+1}/{EPOCHS}")

        for step_idx in progress_bar:
            global_step += 1
           patient_id = real_patients[step_idx % len(real_patients)]

            # --- 1. Stream 3D Volume into RAM ---
            for patient_dir in streamer.stream_patient(patient_id):
                volume_tensor = load_patient_3d_volume(patient_dir)

                # --- 2. Extract 3D Visual Embeddings ---
                with torch.no_grad():
                    image_embs = brainiac_encoder(volume_tensor)

               # --- 3. Retrieve Questions & Formulate Prompt ---
                qa_list = real_qas.get(patient_id)
                qa_pair = qa_list[0]
                
                # Format the FULL text (Prompt + Answer)
                full_text = format_llama3_prompt(qa_pair['question'], qa_pair['answer'])
                tokenized = model.tokenizer(full_text, return_tensors="pt", padding=False).to("cuda")
                
                # Format ONLY the Prompt (No Answer) to find the exact token cutoff
                prompt_only = format_llama3_prompt(qa_pair['question'])
                prompt_tokens = model.tokenizer(prompt_only, return_tensors="pt", padding=False).input_ids
                prompt_length = prompt_tokens.shape[1]
                
                # --- 4. Apply Mathematically Precise Loss Masking ---
                labels = tokenized.input_ids.clone()
                
                # Mask everything in the prompt so loss is ONLY calculated on the answer
                labels[0, :prompt_length] = -100 

                # --- 5. Forward Pass & Optimization ---
                loss = model(image_embs, tokenized, labels)
                loss.backward()
                
                # Clip gradients to prevent explosion
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                optimizer.zero_grad()
                
                current_loss = loss.item()
                epoch_loss += current_loss
                progress_bar.set_postfix({"loss": f"{current_loss:.4f}"})









                
                # Checkpoint saving logic every N steps
                if global_step % CHECKPOINT_EVERY_STEPS == 0:
                    save_vlm_checkpoint(model, optimizer, epoch, global_step, current_loss, "latest_checkpoint.pt")
                    if current_loss < best_loss:
                        best_loss = current_loss
                        save_vlm_checkpoint(model, optimizer, epoch, global_step, current_loss, "best_checkpoint.pt")
                    check_kaggle_uptime()

        avg_epoch_loss = epoch_loss / STEPS_PER_EPOCH
        print(f"\n✅ Epoch {epoch+1} Completed | Average Loss: {avg_epoch_loss:.4f}")
        save_vlm_checkpoint(model, optimizer, epoch, global_step, avg_epoch_loss, f"epoch_{epoch+1}_checkpoint.pt")
        check_kaggle_uptime()

    # Save Final Artifacts
    final_dir = "/kaggle/working/brain_tumor_vlm_final"
    os.makedirs(final_dir, exist_ok=True)
    torch.save(model.adapter.state_dict(), os.path.join(final_dir, "adapter.pt"))
    model.llm.save_pretrained(os.path.join(final_dir, "lora"))
    print(f"\n🏆 TRAINING FINISHED SUCCESSFULLY! Final weights exported to: {final_dir}")

# EXECUTE TRAINING PIPELINE
run_full_training_pipeline()